In [75]:
!pip install ctgan --quiet

In [76]:
import time
import json
import warnings
import random 
import torch
import dataclasses
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List

import wandb
import lightgbm as lgb
from ctgan import CTGAN
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, classification_report,
    matthews_corrcoef,
)
from imblearn.metrics import geometric_mean_score
from scipy.spatial.distance import jensenshannon
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Dependencies loaded.')

In [77]:
wandb.login(key='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')


In [ ]:
import os
from pathlib import Path

# ── Works locally AND on Colab/server without any changes ─────────────────
# Local:  checks ieee_feature_store/ at project root first; uses it if found
# Colab:  falls back to wget from GitHub if parquets are not present

if os.path.exists("/content"):
    STORE = Path("/content/ieee_feature_store")
else:
    _HERE = Path.cwd()
    # notebook lives at  <project_root>/notebooks/ieee/
    # feature store is at <project_root>/ieee_feature_store/
    _candidate = _HERE.parent.parent / "ieee_feature_store"
    STORE = _candidate if _candidate.exists() else _HERE / "ieee_feature_store"

STORE.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/kkipngenokoech/synthesize-or-reconstruct/main/ieee_feature_store"

for fname in ["pipeline_a_train.parquet", "pipeline_a_val.parquet",
              "pipeline_b_train.parquet", "holdout.parquet"]:
    out = STORE / fname
    if out.exists() and out.stat().st_size > 0:
        print(f"Found locally  : {fname}  ({out.stat().st_size / 1e6:.1f} MB)")
    else:
        if out.exists(): out.unlink()
        print(f"Downloading    : {fname} ...")
        ret = os.system(f'wget -q --show-progress -O "{out}" "{BASE}/{fname}"')
        if ret != 0 or out.stat().st_size == 0:
            out.unlink(missing_ok=True)
            raise RuntimeError(f"Download failed for {fname}. "
                               "Run ieee/pipeline.ipynb first and commit the parquets.")
        print(f"Downloaded     : {fname}  ({out.stat().st_size / 1e6:.1f} MB)")

print(f"\nIEEE feature store ready at: {STORE.resolve()}")
print("Files:", sorted(p.name for p in STORE.glob("*.parquet")))


In [79]:
STORE = Path("/content/ieee_feature_store")

a_train  = pd.read_parquet(STORE / 'pipeline_a_train.parquet')
a_val    = pd.read_parquet(STORE / 'pipeline_a_val.parquet')
holdout  = pd.read_parquet(STORE / 'holdout.parquet')
 
assert list(a_train.columns) == list(a_val.columns) == list(holdout.columns), \
    "Column mismatch between train, val, and holdout partitions."
 
print(f'Pipeline A train shape : {a_train.shape}')
print(f'Pipeline A val shape   : {a_val.shape}')
print(f'Fraud holdout shape    : {holdout.shape}')
print(f'Pipeline A fraud rate  : {a_train["Class"].mean():.4%}')  # natural ~0.17%
print(f'Feature count          : {a_train.shape[1]}')              # 31 columns
 
# Verify zero overlap between training data and holdout
_train_ids   = set(zip(a_train['Time'], a_train['Amount'], a_train['V1']))
_holdout_ids = set(zip(holdout['Time'], holdout['Amount'], holdout['V1']))
_overlap     = len(_train_ids & _holdout_ids)
print(f'Train-holdout overlap  : {_overlap} records (must be 0)')
assert _overlap == 0, f'CRITICAL: {_overlap} records in both a_train and holdout.'
print('Overlap check          : PASSED')

In [80]:
@dataclass
class CTGANConfig:
    target_col: str = 'Class'
    n_features: int = 15

    # ── PATCHED CTGAN architecture ────────────────────────────────────────
    embedding_dim: int = 128
    generator_dim: tuple = (256, 256)
    discriminator_dim: tuple = (256, 256)
    generator_lr: float = 2e-4
    generator_decay: float = 1e-6
    discriminator_lr: float = 2e-4
    discriminator_decay: float = 1e-6
    batch_size: int = 100       
    discriminator_steps: int = 1  
    log_frequency: bool = True
    pac: int = 10                 

    n_epochs: int = 800           

    # ── LightGBM (unchanged) ─────────────────────────────────────────────
    lgb_num_leaves: int = 63
    lgb_learning_rate: float = 0.05
    lgb_n_estimators: int = 500
    lgb_min_child_samples: int = 20
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8
    lgb_reg_alpha: float = 0.1
    lgb_reg_lambda: float = 0.1
    lgb_early_stopping_rounds: int = 50

    # ── PATCHED synthetic budget ──────────────────────────────────────────
    # Will be set dynamically below based on N_real_fraud. Do not hardcode 5000.
    n_synthetic_fraud: int = 730  # placeholder; overridden below

    # ── Quality gate thresholds ───────────────────────────────────────────
    mmd_threshold: float = 0.05
    jsd_threshold: float = 0.15   
    novelty_min_distance: float = 0.01
    disc_auc_threshold: float = 0.85
    

# FEATURE_COLS defines the 15 model features used for CTGAN and LightGBM.
# Time is excluded: it is a temporal split key, not a fraud signal.
FEATURE_COLS = [f'C{i}' for i in range(1, 15)] + ['TransactionAmt']
 
cfg            = CTGANConfig()
cfg.n_features = len(FEATURE_COLS)   # 15
 
print(f'Config: embedding_dim={cfg.embedding_dim}, n_epochs={cfg.n_epochs}, '
      f'batch_size={cfg.batch_size}, n_features={cfg.n_features}')

In [81]:
run = wandb.init(
    project = 'PRINCIPLES AND ENGINEERING APPLICATIONS OF AI',
    name    = 'fm-ctgan-pipeline-a-run-2',
    tags    = ['CTGAN', 'Pipeline-A', 'LightGBM'],
    config  = asdict(cfg),         # logs every param in CTGANConfig
)

# Sync config back so wandb sweeps can override values
wcfg = wandb.config
cfg.embedding_dim           = wcfg.embedding_dim
cfg.batch_size              = wcfg.batch_size
cfg.n_epochs                = wcfg.n_epochs
cfg.generator_lr            = wcfg.generator_lr
cfg.discriminator_lr        = wcfg.discriminator_lr
cfg.discriminator_steps     = wcfg.discriminator_steps
cfg.pac                     = wcfg.pac
cfg.n_synthetic_fraud       = wcfg.n_synthetic_fraud
cfg.lgb_num_leaves          = wcfg.lgb_num_leaves
cfg.lgb_learning_rate       = wcfg.lgb_learning_rate
cfg.lgb_n_estimators        = wcfg.lgb_n_estimators
cfg.lgb_subsample           = wcfg.lgb_subsample
cfg.lgb_colsample_bytree    = wcfg.lgb_colsample_bytree
cfg.lgb_reg_alpha           = wcfg.lgb_reg_alpha
cfg.lgb_reg_lambda          = wcfg.lgb_reg_lambda
cfg.generator_decay         = wcfg.generator_decay
cfg.discriminator_decay     = wcfg.discriminator_decay

print('W&B run initialised:', run.name)
print('W&B project URL    :', run.url)

In [82]:
# FEATURE_COLS is already defined above.
# CTGAN trains on continuous features only -- no categorical columns.
CATEGORICAL_COLS = []
 
# Extract fraud records and select model features only (no Time, no Class).
# CTGAN learns the fraud feature distribution from these 29 features.
fraud_train_ctgan = (
    a_train[a_train[cfg.target_col] == 1][FEATURE_COLS]
    .reset_index(drop=True)
)
 
print(f'Fraud records for CTGAN training : {len(fraud_train_ctgan)}')
print(f'CTGAN training features          : {len(FEATURE_COLS)} continuous')
print(f'Categorical features             : {len(CATEGORICAL_COLS)} (none)')
print(f'Training data shape              : {fraud_train_ctgan.shape}')

In [83]:
N_REAL_FRAUD = len(fraud_train_ctgan)

# Principled synthetic budget: 2x real, capped at 1500 for small-sample safety
SYNTHETIC_EXPANSION_FACTOR = 3.0
N_SYNTHETIC_TARGET = int(N_REAL_FRAUD * SYNTHETIC_EXPANSION_FACTOR)
cfg.n_synthetic_fraud = N_SYNTHETIC_TARGET

print(f'Real fraud records       : {N_REAL_FRAUD}')
print(f'Synthetic budget (2x cap): {cfg.n_synthetic_fraud}')
print(f'Expansion factor         : {cfg.n_synthetic_fraud / N_REAL_FRAUD:.1f}x')

In [84]:
# # ── Step 1: Initialise CTGAN ──────────────────────────────────────────────

# ctgan_model = CTGAN(
#     embedding_dim        = cfg.embedding_dim,
#     generator_dim        = cfg.generator_dim,
#     discriminator_dim    = cfg.discriminator_dim,
#     generator_lr         = cfg.generator_lr,
#     generator_decay      = cfg.generator_decay,
#     discriminator_lr     = cfg.discriminator_lr,
#     discriminator_decay  = cfg.discriminator_decay,
#     batch_size           = cfg.batch_size,
#     discriminator_steps  = cfg.discriminator_steps,
#     log_frequency        = cfg.log_frequency,
#     pac                  = cfg.pac,
#     epochs               = cfg.n_epochs,
#     verbose              = True,           # prints per-epoch losses
# )

# print('CTGAN model initialised.')
# print(f'  Generator dims    : {cfg.generator_dim}')
# print(f'  Discriminator dims: {cfg.discriminator_dim}')
# print(f'  Embedding dim     : {cfg.embedding_dim}')

In [85]:
# # ── Simpler alternative if the subclass above raises AttributeError ───────

# print('Starting CTGAN training...')
# t0 = time.time()

# ctgan_model.fit(
#     fraud_train_ctgan,
#     discrete_columns=CATEGORICAL_COLS
# )

# train_time_min = (time.time() - t0) / 60
# print(f'Training complete in {train_time_min:.1f} min')

# # Log summary training metrics (CTGAN stores loss history internally)
# # Access via ctgan_model.loss_values if available in your version
# if hasattr(ctgan_model, 'loss_values'):
#     loss_df = ctgan_model.loss_values   # DataFrame: Epoch, Generator Loss, Discriminator Loss
#     for _, row in loss_df.iterrows():
#         wandb.log({
#             'train/epoch'  : int(row['Epoch']),
#             'train/g_loss' : float(row['Generator Loss']),
#             'train/d_loss' : float(row['Discriminator Loss']),
#         }, step=int(row['Epoch']))
#     print(f'Logged {len(loss_df)} epochs to W&B.')
# else:
#     wandb.log({'train/train_time_min': train_time_min})
#     print('loss_values not available in this CTGAN version — logged runtime only.')

## Synthetic Data Generation


### Synthetic Quality Gates (matches other experimenrt exactly)


In [89]:
# --- Quality helper functions ---
from sklearn.model_selection import StratifiedKFold

def _compute_mmd(real, synthetic, gamma=1.0):
    cap = 500
    rng = np.random.default_rng(42)
    r = real[rng.choice(len(real), min(cap, len(real)), replace=False)]
    s = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    def rbf(A, B):
        return np.exp(-gamma * np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)).mean()
    return float(rbf(r, r) - 2 * rbf(r, s) + rbf(s, s))

def _compute_jsd(real, synthetic, n_bins=50):
    scores = []
    for i in range(real.shape[1]):
        combined = np.concatenate([real[:, i], synthetic[:, i]])
        bins = np.linspace(combined.min(), combined.max(), n_bins + 1)
        r_prob = np.histogram(real[:, i], bins=bins, density=True)[0]
        s_prob = np.histogram(synthetic[:, i], bins=bins, density=True)[0]
        r_prob /= r_prob.sum() + 1e-12
        s_prob /= s_prob.sum() + 1e-12
        scores.append(float(jensenshannon(r_prob, s_prob)))
    return float(np.mean(scores))

def _novelty_check(real, synthetic, min_dist=0.01, cap=200):
    rng = np.random.default_rng(42)
    r = real[rng.choice(len(real), min(cap, len(real)), replace=False)]
    s = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    dists = np.sqrt(np.sum((s[:, None, :] - r[None, :, :]) ** 2, axis=-1))
    return float(dists.mean()) >= min_dist, float(dists.mean())

def _discriminability_check(real, synthetic, auc_threshold=0.75, n_splits=5):
    X = np.vstack([real, synthetic])
    y = np.array([0] * len(real) + [1] * len(synthetic))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y[train_idx]
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_val_sc = scaler.transform(X_val)
        clf = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
        clf.fit(X_train_sc, y_train)
        oof_preds[val_idx] = clf.predict_proba(X_val_sc)[:, 1]
    auc = float(roc_auc_score(y, oof_preds))
    return auc <= auc_threshold, auc


# --- CTGAN retry loop ---
# Quality gates are INFORMATIONAL ONLY per paper Section V.B:
# all three Pipeline A GANs fail JSD and discriminator AUC gates yet achieve
# AUROC >= 0.9998. Gate compliance does not predict downstream classifier utility.
SEEDS = [42, 52, 62]
accepted = False
best_attempt = None

for attempt, seed in enumerate(SEEDS, 1):
    print(f"\n=== CTGAN attempt {attempt}/{len(SEEDS)} | seed={seed} ===")
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    ctgan_model = CTGAN(
        embedding_dim=cfg.embedding_dim,
        generator_dim=cfg.generator_dim,
        discriminator_dim=cfg.discriminator_dim,
        generator_lr=cfg.generator_lr,
        generator_decay=cfg.generator_decay,
        discriminator_lr=cfg.discriminator_lr,
        discriminator_decay=cfg.discriminator_decay,
        batch_size=cfg.batch_size,
        discriminator_steps=cfg.discriminator_steps,
        log_frequency=cfg.log_frequency,
        pac=cfg.pac,
        epochs=cfg.n_epochs,
        verbose=True,
    )

    ctgan_model.fit(fraud_train_ctgan, discrete_columns=[])

    synthetic_raw = ctgan_model.sample(cfg.n_synthetic_fraud)
    synthetic_raw["Class"] = 1
    synthetic_fraud_candidate = synthetic_raw[FEATURE_COLS + ["Class"]].copy()
    synthetic_fraud_candidate["Amount"] = synthetic_fraud_candidate["Amount"].clip(lower=0)

    real_arr = fraud_train_ctgan[FEATURE_COLS].values.astype(np.float64)
    syn_arr  = synthetic_fraud_candidate[FEATURE_COLS].values.astype(np.float64)

    mmd = _compute_mmd(real_arr, syn_arr)
    jsd = _compute_jsd(real_arr, syn_arr)
    nov_pass, nov = _novelty_check(real_arr, syn_arr, min_dist=cfg.novelty_min_distance)
    disc_pass, dauc = _discriminability_check(real_arr, syn_arr, auc_threshold=cfg.disc_auc_threshold)

    mmd_pass = mmd <= cfg.mmd_threshold
    jsd_pass = jsd <= cfg.jsd_threshold
    overall = all([mmd_pass, jsd_pass, nov_pass, disc_pass])

    print(
        f"  MMD={mmd:.4f} ({'PASS' if mmd_pass else 'FAIL'}) | "
        f"JSD={jsd:.4f} ({'PASS' if jsd_pass else 'FAIL'}) | "
        f"DiscAUC={dauc:.4f} ({'PASS' if disc_pass else 'FAIL'}) | "
        f"Novelty={'PASS' if nov_pass else 'FAIL'} | "
        f"Gates={'PASS' if overall else 'FAIL (informational)'}"
    )

    wandb.log({
        "attempt": attempt, "seed": seed,
        "quality_gates/mmd": mmd, "quality_gates/jsd_mean": jsd,
        "quality_gates/novelty_min_dist": nov,
        "quality_gates/discriminator_auc": dauc,
        "quality_gates/overall_pass": int(overall),
    })

    score = dauc + jsd  # lower = better
    if best_attempt is None or score < best_attempt["score"]:
        best_attempt = {
            "score": score, "seed": seed,
            "mmd": mmd, "jsd": jsd, "nov": nov, "dauc": dauc,
            "synthetic_fraud": synthetic_fraud_candidate.copy(),
            "gates_passed": overall,
        }

    if overall:
        accepted = True
        print(f"  All quality gates passed on attempt {attempt}.")
        break

# Quality gates are informational: always use the best synthetic data found.
# This matches the finding that gate compliance != downstream classifier quality.
USE_CTGAN = True
if accepted:
    synthetic_fraud = best_attempt["synthetic_fraud"].copy()
    gate_summary = {
        "mmd": best_attempt["mmd"], "jsd": best_attempt["jsd"],
        "nov": best_attempt["nov"], "dauc": best_attempt["dauc"],
        "passed": True,
    }
    print(f"\nUsing quality-gate-passing synthetic data (seed={best_attempt['seed']}).")
else:
    synthetic_fraud = best_attempt["synthetic_fraud"].copy()
    gate_summary = {
        "mmd": best_attempt["mmd"], "jsd": best_attempt["jsd"],
        "nov": best_attempt["nov"], "dauc": best_attempt["dauc"],
        "passed": False,
    }
    print(
        f"\n[INFO] No attempt passed all quality gates. "
        f"Using best attempt (seed={best_attempt['seed']}, score={best_attempt['score']:.4f}). "
        f"Gates are informational per paper Section V.B."
    )

wandb.log({"ctgan/use_synthetic": int(USE_CTGAN), "ctgan/gates_passed": int(accepted)})
print(f"Synthetic fraud samples to augment: {len(synthetic_fraud)}")


In [90]:
# Align both DataFrames on model columns before concatenation.
# a_train has 31 columns (includes Time); synthetic_fraud has 30 (no Time).
a_train_model = a_train[FEATURE_COLS + ["Class"]]

if USE_CTGAN and len(synthetic_fraud) > 0:
    augmented_train = pd.concat(
        [a_train_model, synthetic_fraud[FEATURE_COLS + ["Class"]]],
        ignore_index=True
    )
else:
    augmented_train = a_train_model.copy()  # fallback baseline

augmented_train = augmented_train.sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)

X_train_lgb = augmented_train[FEATURE_COLS]
y_train_lgb = augmented_train["Class"]
X_val_lgb   = a_val[FEATURE_COLS]
y_val_lgb   = a_val["Class"]

print(f"USE_CTGAN={USE_CTGAN} | Augmented train: {augmented_train.shape} | Fraud rate: {y_train_lgb.mean():.4%}")

aug_fraud_rate = y_train_lgb.mean()
 
wandb.log({
    'lgb/augmented_train_size': len(augmented_train),
    'lgb/augmented_fraud_rate': aug_fraud_rate,
    'lgb/is_unbalance': 1,
})

In [91]:
# ── LightGBM Classifier ───────────────────────────────────────────────────
# Metric: 'average_precision' (AUPRC) only.
# Using two metrics ([average_precision, auc]) caused early stopping to fire
# at round 1 because val AUC peaked immediately with is_unbalance=True on
# extreme imbalance. AUPRC improves steadily over training and is the correct
# metric for imbalanced data where base rate matters.
lgb_params = {
    'objective'         : 'binary',
    'metric'            : ['average_precision'],
    'boosting_type'     : 'gbdt',
    'num_leaves'        : cfg.lgb_num_leaves,
    'max_depth'         : -1,
    'learning_rate'     : cfg.lgb_learning_rate,
    'n_estimators'      : cfg.lgb_n_estimators,
    'min_child_samples' : cfg.lgb_min_child_samples,
    'subsample'         : cfg.lgb_subsample,
    'colsample_bytree'  : cfg.lgb_colsample_bytree,
    'reg_alpha'         : cfg.lgb_reg_alpha,
    'reg_lambda'        : cfg.lgb_reg_lambda,
    'is_unbalance'      : True,
    'random_state'      : GLOBAL_SEED,
    'n_jobs'            : -1,
    'verbose'           : -1,
}

dtrain = lgb.Dataset(X_train_lgb, label=y_train_lgb)
dval   = lgb.Dataset(X_val_lgb,   label=y_val_lgb, reference=dtrain)

print('Training LightGBM...')
print(f'  Train: {X_train_lgb.shape} | fraud rate: {y_train_lgb.mean():.4%}')
print(f'  Val  : {X_val_lgb.shape}   | fraud rate: {y_val_lgb.mean():.4%}')
t0 = time.time()

lgb_model = lgb.train(
    lgb_params, dtrain,
    num_boost_round = cfg.lgb_n_estimators,
    valid_sets      = [dtrain, dval],
    valid_names     = ['train', 'val'],
    callbacks       = [
        lgb.early_stopping(stopping_rounds=cfg.lgb_early_stopping_rounds, verbose=True),
        lgb.log_evaluation(period=50),
    ]
)

lgb_time = time.time() - t0
print(f'LightGBM complete in {lgb_time:.1f}s | Best iteration: {lgb_model.best_iteration}')


In [92]:
# ── Step 1: Threshold selection on a_val (validation set only) ─────────────
val_probs   = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)
best_thresh = 0.5
best_f1     = 0.0
thresh_rows = []

for thresh in np.arange(0.01, 0.99, 0.005):
    preds = (val_probs >= thresh).astype(int)
    f1_t  = f1_score(y_val_lgb, preds, zero_division=0)
    p     = precision_score(y_val_lgb, preds, zero_division=0)
    r     = recall_score(y_val_lgb, preds, zero_division=0)
    thresh_rows.append({'threshold': round(thresh, 4), 'f1': f1_t, 'precision': p, 'recall': r})
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, thresh

wandb.log({'eval/threshold_sweep': wandb.Table(dataframe=pd.DataFrame(thresh_rows))})
print(f'Best threshold (a_val): {best_thresh:.4f}  (val F1={best_f1:.4f})')

# ── Step 2: Holdout evaluation at natural class distribution ───────────────
X_holdout_eval = holdout[FEATURE_COLS]
y_holdout_eval = holdout['Class']

print(f'\nHoldout set: {len(holdout):,} records | '
      f'Fraud: {y_holdout_eval.sum()} ({y_holdout_eval.mean():.4%}) | '
      f'Normal: {(y_holdout_eval == 0).sum():,}')
print('Natural class distribution -- no artificial fraud rate inflation.')

t_infer           = time.time()
holdout_probs     = lgb_model.predict(X_holdout_eval, num_iteration=lgb_model.best_iteration)
inference_time_ms = (time.time() - t_infer) * 1000
holdout_preds     = (holdout_probs >= best_thresh).astype(int)

f1        = f1_score(y_holdout_eval,        holdout_preds, zero_division=0)
precision = precision_score(y_holdout_eval, holdout_preds, zero_division=0)
recall    = recall_score(y_holdout_eval,    holdout_preds, zero_division=0)
auroc     = roc_auc_score(y_holdout_eval,   holdout_probs)
auprc     = average_precision_score(y_holdout_eval, holdout_probs)
mcc       = matthews_corrcoef(y_holdout_eval, holdout_preds)
gmean     = geometric_mean_score(y_holdout_eval, holdout_preds)

print('\n' + '=' * 55)
print(' CTGAN + LightGBM -- Holdout Evaluation Results')
print('=' * 55)
print(f' F1-Score  : {f1:.4f}')
print(f' Precision : {precision:.4f}')
print(f' Recall    : {recall:.4f}')
print(f' AUROC     : {auroc:.4f}')
print(f' AUPRC     : {auprc:.4f}')
print(f' MCC       : {mcc:.4f}')
print(f' G-mean    : {gmean:.4f}')
print(f' Inference : {inference_time_ms:.2f} ms ({len(X_holdout_eval):,} records)')
print(f' Threshold : {best_thresh:.4f}')
print('=' * 55)
print(classification_report(y_holdout_eval, holdout_preds, target_names=['Legit', 'Fraud']))

wandb.log({
    'holdout/f1'               : f1,
    'holdout/precision'        : precision,
    'holdout/recall'           : recall,
    'holdout/auroc'            : auroc,
    'holdout/auprc'            : auprc,
    'holdout/mcc'              : mcc,
    'holdout/gmean'            : gmean,
    'holdout/inference_time_ms': inference_time_ms,
    'holdout/threshold'        : best_thresh,
    'holdout/n_records'        : len(X_holdout_eval),
    'holdout/fraud_rate'       : float(y_holdout_eval.mean()),
})

wandb.run.summary['best_f1']        = f1
wandb.run.summary['best_auroc']     = auroc
wandb.run.summary['best_auprc']     = auprc
wandb.run.summary['best_recall']    = recall
wandb.run.summary['best_mcc']       = mcc
wandb.run.summary['best_gmean']     = gmean

In [94]:
results = {
    'model'               : 'CTGAN + LightGBM',
    'pipeline'            : 'A',
    'author'              : 'Francis Waithaka (fwaithak)',
    'f1'                  : round(f1, 4),
    'precision'           : round(precision, 4),
    'recall'              : round(recall, 4),
    'auroc'               : round(auroc, 4),
    'auprc'               : round(auprc, 4),
    'mcc'                 : round(mcc, 4),
    'gmean'               : round(gmean, 4),
    'inference_time_ms'   : round(inference_time_ms, 2),
    'threshold'           : round(best_thresh, 4),
    'n_synthetic_fraud'   : cfg.n_synthetic_fraud,
    'quality_gates_passed': gate_summary["passed"],
    'quality_gate_detail' : {
        'mmd'         : round(gate_summary["mmd"], 6),
        'jsd'         : round(gate_summary["jsd"], 6),
        'novelty_dist': round(gate_summary["nov"], 6),
        'disc_auc'    : round(gate_summary["dauc"], 4),
    },
    'used_ctgan_synthetic': USE_CTGAN,
    'wandb_run_url'       : wandb.run.url,
}

out_path = Path('/content/ieee_cieee_tgan_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

wandb.save(str(out_path))
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))

wandb.run.summary['best_f1']    = f1
wandb.run.summary['best_auroc'] = auroc
wandb.run.summary['best_recall'] = recall
wandb.run.summary['best_mcc']   = mcc
wandb.run.summary['best_gmean'] = gmean

wandb.finish()
print('W&B run finished.')